# 65 - Late Fusion TL across All Benchmarks

Menambahkan **Late Fusion TL** (ResNet18 CNN_TL + FCNN) yang sebelumnya terlewat di benchmark notebooks (36, 37, 60, 61, 62).

**Strategi**:
1. Reuse checkpoint `CNN_TL_B1/model.pth` dan `FCNN_B1/model.pth` kalau sudah ada
2. Train dari nol kalau belum ada (fallback)
3. Late Fusion = weighted softmax averaging (tune weight di val, eval di test)
4. Bonus: fill missing `micro_f1` = `accuracy` di JSON existing (multi-class single-label: identical)

**Output**: update `{dataset}_{num_class}c_results.json` dengan key `Late_Fusion_TL_B1`.

**Datasets**: CK+ 7c/4c, JAFFE 7c/4c, RAF-DB 7c/4c, KDEF 7c/4c, Primer 7c/4c — **10 benchmark variants**.

In [ ]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionCNNTransfer, EmotionFCNN
from training.utils import train_model, full_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
LR_TL = 0.00005
LR = 0.0001

BENCHMARK_DIR = PROJECT_ROOT / 'data' / 'benchmark'
PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
MODELS_DIR = PROJECT_ROOT / 'models' / 'benchmark'

REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)

print('Setup complete.')

In [ ]:
# ── Helpers ──

def _subject_split(subjects, seed=42, train_ratio=0.8, val_ratio=0.1):
    rng = np.random.RandomState(seed)
    uniq = np.array(sorted(set(subjects.tolist())))
    rng.shuffle(uniq)
    n = len(uniq)
    n_tr = int(n * train_ratio)
    n_v = int(n * val_ratio)
    return set(uniq[:n_tr].tolist()), set(uniq[n_tr:n_tr+n_v].tolist()), set(uniq[n_tr+n_v:].tolist())


def load_dataset(dataset_name, num_classes):
    """Return (tr_img, tr_lm, tr_y, v_img, v_lm, v_y, te_img, te_lm, te_y)."""
    if dataset_name == 'rafdb':
        d = BENCHMARK_DIR / f'rafdb_{num_classes}class'
        X = np.load(d / 'X_train_images.npy')
        L = np.load(d / 'X_train_landmarks.npy')
        y = np.load(d / 'y_train.npy')
        te_X = np.load(d / 'X_test_images.npy')
        te_L = np.load(d / 'X_test_landmarks.npy')
        te_y = np.load(d / 'y_test.npy')
        idx_tr, idx_v = train_test_split(np.arange(len(y)), test_size=0.1, stratify=y, random_state=42)
        return (X[idx_tr], L[idx_tr], y[idx_tr],
                X[idx_v], L[idx_v], y[idx_v],
                te_X, te_L, te_y)

    if dataset_name == 'kdef':
        d = BENCHMARK_DIR / f'kdef_{num_classes}class'
        return (np.load(d / 'X_train_images.npy'),
                np.load(d / 'X_train_landmarks.npy'),
                np.load(d / 'y_train.npy'),
                np.load(d / 'X_val_images.npy'),
                np.load(d / 'X_val_landmarks.npy'),
                np.load(d / 'y_val.npy'),
                np.load(d / 'X_test_images.npy'),
                np.load(d / 'X_test_landmarks.npy'),
                np.load(d / 'y_test.npy'))

    if dataset_name == 'primer':
        tr_X = np.load(PRIMER_DIR / 'X_train_images.npy')
        tr_L = np.load(PRIMER_DIR / 'X_train_landmarks.npy')
        tr_y = np.load(PRIMER_DIR / 'y_train.npy')
        v_X = np.load(PRIMER_DIR / 'X_val_images.npy')
        v_L = np.load(PRIMER_DIR / 'X_val_landmarks.npy')
        v_y = np.load(PRIMER_DIR / 'y_val.npy')
        te_X = np.load(PRIMER_DIR / 'X_test_images.npy')
        te_L = np.load(PRIMER_DIR / 'X_test_landmarks.npy')
        te_y = np.load(PRIMER_DIR / 'y_test.npy')
        if num_classes == 4:
            tr_y = REMAP_4[tr_y]
            v_y = REMAP_4[v_y]
            te_y = REMAP_4[te_y]
        return tr_X, tr_L, tr_y, v_X, v_L, v_y, te_X, te_L, te_y

    if dataset_name in ('ckplus', 'jaffe'):
        # Old-style benchmark: X_images.npy, subjects.npy (single split file)
        if dataset_name == 'ckplus' and num_classes == 4:
            d = BENCHMARK_DIR / 'ckplus_4class_contempt'  # includes contempt
        else:
            d = BENCHMARK_DIR / f'{dataset_name}_{num_classes}class'
        X = np.load(d / 'X_images.npy')
        L = np.load(d / 'X_landmarks.npy')
        y = np.load(d / 'y_labels.npy')
        subjects = np.load(d / 'subjects.npy', allow_pickle=True)
        tr_subs, v_subs, te_subs = _subject_split(subjects)
        tr_idx = np.where(np.isin(subjects, list(tr_subs)))[0]
        v_idx = np.where(np.isin(subjects, list(v_subs)))[0]
        te_idx = np.where(np.isin(subjects, list(te_subs)))[0]
        return (X[tr_idx], L[tr_idx], y[tr_idx],
                X[v_idx], L[v_idx], y[v_idx],
                X[te_idx], L[te_idx], y[te_idx])

    raise ValueError(f'Unknown dataset {dataset_name}')


def checkpoint_dir(dataset_name, num_classes, model_key):
    """Checkpoint path. Handles naming convention differences:
    - rafdb/kdef/primer: models/benchmark/{ds}/{num}c/{key}/
    - ckplus/jaffe:      models/benchmark/{ds}/{ds}_{num}c/{key}/  (nb 36/37 style)
    """
    if dataset_name in ('ckplus', 'jaffe'):
        return MODELS_DIR / dataset_name / f'{dataset_name}_{num_classes}c' / model_key
    return MODELS_DIR / dataset_name / f'{num_classes}c' / model_key


def results_file(dataset_name, num_classes):
    return MODELS_DIR / dataset_name / f'{dataset_name}_{num_classes}c_results.json'


def make_loader(img, lm, y, model_type, batch_size=BATCH_SIZE, shuffle=True):
    img_t = torch.from_numpy(img).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(lm)
    y_t = torch.from_numpy(y).long()
    if model_type == 'cnn':
        ds = TensorDataset(img_t, y_t)
    else:
        ds = TensorDataset(lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def ensure_checkpoint(dataset_name, num_classes, model_key, model_class, model_type, lr,
                     tr_img, tr_lm, tr_y, v_img, v_lm, v_y):
    save_dir = checkpoint_dir(dataset_name, num_classes, model_key)
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / 'model.pth'

    model = model_class(num_classes=num_classes).to(device)
    if save_path.exists():
        print(f'    [SKIP train] loading existing {save_path}')
        model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
        return model

    print(f'    [TRAIN] {model_key} ...')
    tr_loader = make_loader(tr_img, tr_lm, tr_y, model_type, BATCH_SIZE, True)
    val_loader = make_loader(v_img, v_lm, v_y, model_type, BATCH_SIZE, False)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    train_model(model, tr_loader, val_loader, crit, opt, sch,
                device, model_type, EPOCHS, PATIENCE, str(save_path))
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    return model


def metrics_triple(y_true, y_pred):
    return {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }


@torch.no_grad()
def batched_softmax(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        x = batch[0].to(device)
        probs.append(torch.softmax(model(x), dim=1).cpu().numpy())
    return np.concatenate(probs, axis=0)


def fill_missing_micro_f1(results_path):
    """For multi-class single-label, micro_f1 = accuracy.
    Fill it in JSON for entries missing `micro_f1` but having `accuracy`.
    """
    if not results_path.exists():
        return 0
    with open(results_path) as f:
        data = json.load(f)
    updated = 0
    for key, r in data.items():
        if isinstance(r, dict) and 'accuracy' in r and 'micro_f1' not in r:
            r['micro_f1'] = r['accuracy']
            updated += 1
    if updated > 0:
        with open(results_path, 'w') as f:
            json.dump(data, f, indent=2)
    return updated


def late_fusion_tl(dataset_name, num_classes):
    print(f"\n{'='*70}")
    print(f'  Late Fusion TL — {dataset_name.upper()} {num_classes}c')
    print(f"{'='*70}")

    # Fill missing micro_f1 first (for existing 5 models in this JSON)
    rf = results_file(dataset_name, num_classes)
    n_filled = fill_missing_micro_f1(rf)
    if n_filled:
        print(f'  [micro_f1 fill] filled {n_filled} missing entries in {rf.name}')

    tr_img, tr_lm, tr_y, v_img, v_lm, v_y, te_img, te_lm, te_y = load_dataset(dataset_name, num_classes)
    print(f'  Train={len(tr_y)}  Val={len(v_y)}  Test={len(te_y)}')

    cnn_tl = ensure_checkpoint(dataset_name, num_classes, 'CNN_TL_B1',
                                EmotionCNNTransfer, 'cnn', LR_TL,
                                tr_img, tr_lm, tr_y, v_img, v_lm, v_y)
    fcnn = ensure_checkpoint(dataset_name, num_classes, 'FCNN_B1',
                              EmotionFCNN, 'fcnn', LR,
                              tr_img, tr_lm, tr_y, v_img, v_lm, v_y)

    v_cnn_loader = make_loader(v_img, v_lm, v_y, 'cnn', BATCH_SIZE, False)
    v_fcnn_loader = make_loader(v_img, v_lm, v_y, 'fcnn', BATCH_SIZE, False)
    te_cnn_loader = make_loader(te_img, te_lm, te_y, 'cnn', BATCH_SIZE, False)
    te_fcnn_loader = make_loader(te_img, te_lm, te_y, 'fcnn', BATCH_SIZE, False)

    vc = batched_softmax(cnn_tl, v_cnn_loader)
    vf = batched_softmax(fcnn, v_fcnn_loader)
    cp = batched_softmax(cnn_tl, te_cnn_loader)
    fp = batched_softmax(fcnn, te_fcnn_loader)

    best_wf1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        pr = (w * vc + (1 - w) * vf).argmax(axis=1)
        f = f1_score(v_y, pr, average='macro', zero_division=0)
        if f > best_wf1:
            best_wf1, best_w = f, w
    print(f'    Best w(CNN_TL)={best_w:.2f}  val macro F1={best_wf1:.4f}')

    preds = (best_w * cp + (1 - best_w) * fp).argmax(axis=1)
    m = metrics_triple(te_y, preds)
    m['best_cnn_tl_weight'] = float(best_w)
    print(f"    Macro={m['macro_f1']:.4f}  Micro={m['micro_f1']:.4f}  Weighted={m['weighted_f1']:.4f}")

    # Update results JSON
    existing = {}
    if rf.exists():
        with open(rf) as f:
            existing = json.load(f)
    existing['Late_Fusion_TL_B1'] = m
    rf.parent.mkdir(parents=True, exist_ok=True)
    with open(rf, 'w') as f:
        json.dump(existing, f, indent=2)
    print(f'    Updated: {rf}')
    return m


print('Helpers ready.')

## Run Late Fusion TL across 5 Datasets × 2 Class Configs

In [ ]:
all_results = {}

# Primer (paling penting)
all_results['primer_7c'] = late_fusion_tl('primer', 7)
all_results['primer_4c'] = late_fusion_tl('primer', 4)

In [ ]:
# CK+
all_results['ckplus_7c'] = late_fusion_tl('ckplus', 7)
all_results['ckplus_4c'] = late_fusion_tl('ckplus', 4)

In [ ]:
# JAFFE
all_results['jaffe_7c'] = late_fusion_tl('jaffe', 7)
all_results['jaffe_4c'] = late_fusion_tl('jaffe', 4)

In [ ]:
# RAF-DB
all_results['rafdb_7c'] = late_fusion_tl('rafdb', 7)
all_results['rafdb_4c'] = late_fusion_tl('rafdb', 4)

In [ ]:
# KDEF
all_results['kdef_7c'] = late_fusion_tl('kdef', 7)
all_results['kdef_4c'] = late_fusion_tl('kdef', 4)

## Ringkasan Late Fusion TL (10 Benchmark Variants)

In [ ]:
print(f"\n{'='*82}")
print(f'  Late Fusion TL — Summary across 10 benchmark variants')
print(f"{'='*82}")
print(f"  {'Dataset':<14} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Acc':>10} {'w(CNN_TL)':>10}")
print(f"  {'-'*72}")
for key, r in all_results.items():
    print(f"  {key:<14} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f} {r.get('best_cnn_tl_weight', 0):>10.2f}")